# Nova seção

In [165]:
!pip install great-expectations

In [166]:
import pandas as pd

df = pd.read_csv("ecommerce_products_raw.csv")

print("Quantidade de registros:", len(df))
print("Quantidade de colunas:", len(df.columns))

df.head()

Quantidade de registros: 120300
Quantidade de colunas: 19


,product_id,title,description,category,subcategory,brand,color,locale,seller_type,price,rating,review_count,stock_quantity,views,orders,units_sold,revenue,created_at,updated_at
0,SKU-0000001,Lumi Smartphones Prata Modelo 236,Produto Lumi Smartphones Prata Modelo 236. Fab...,Eletrônicos,Smartphones,Lumi,Prata,en-US,3P,1191.10,4.0,88,40,858,15,15,17866.50,2025-12-27,2026-06-16
1,SKU-0000002,Vitta Colecionáveis Branco Modelo 240,Produto Vitta Colecionáveis Branco Modelo 240....,Games,Colecionáveis,Vitta,Branco,pt-BR,1P,58.63,4.4,91,37,873,51,52,3048.76,2026-05-06,2026-12-17
2,SKU-0000003,Lumi Ficção Verde Modelo 824,Produto Lumi Ficção Verde Modelo 824. Fabricad...,Livros,Ficção,Lumi,Verde,pt-BR,3P,91.96,4.1,93,44,939,28,29,2666.84,2025-12-23,2026-09-30
3,SKU-0000004,PlayCore Ciclismo Rosa Modelo 737,Produto PlayCore Ciclismo Rosa Modelo 737. Fab...,Esportes,Ciclismo,PlayCore,Rosa,en-US,3P,183.71,4.3,96,49,876,17,20,3674.20,2026-04-18,2027-02-09
4,SKU-0000005,UrbanFit Fitness Preto Modelo 498,Produto UrbanFit Fitness Preto Modelo 498. Fab...,Esportes,Fitness,UrbanFit,Preto,en-US,3P,396.34,3.9,79,43,861,56,62,24573.08,2025-02-11,2025-02-18


In [167]:
quality_report = pd.DataFrame({
    "campo": df.columns,
    "tipo": df.dtypes.astype(str).values,
    "nulos": df.isna().sum().values,
    "percentual_nulos": (df.isna().mean() * 100).round(2).values,
    "valores_distintos": df.nunique().values
})

quality_report.sort_values(
    "percentual_nulos",
    ascending=False
)

,campo,tipo,nulos,percentual_nulos,valores_distintos
9,price,float64,502,0.42,49962
18,updated_at,object,302,0.25,874
1,title,object,200,0.17,117028
2,description,object,0,0.00,117378
0,product_id,object,0,0.00,120000
5,brand,object,0,0.00,12
3,category,object,0,0.00,16
6,color,object,0,0.00,8
7,locale,object,0,0.00,2
8,seller_type,object,0,0.00,3


In [168]:
import great_expectations as gx
import pandas as pd

df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df["stock_quantity"] = pd.to_numeric(df["stock_quantity"], errors="coerce")

df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
df["updated_at"] = pd.to_datetime(df["updated_at"], errors="coerce")


quality_checks = {
    "product_id_nulo": int(df["product_id"].isna().sum()),

    "product_id_duplicado": int(
        df["product_id"].duplicated(keep=False).sum()
    ),

    "title_nulo": int(df["title"].isna().sum()),

    "title_vazio": int(
        df["title"].fillna("").astype(str).str.strip().eq("").sum()
    ),

    "price_nulo": int(df["price"].isna().sum()),

    "price_negativo": int(
        (df["price"].dropna() < 0).sum()
    ),

    "rating_invalido": int(
        (~df["rating"].dropna().between(1, 5)).sum()
    ),

    "stock_negativo": int(
        (df["stock_quantity"].dropna() < 0).sum()
    ),

    "created_at_invalido": int(
        df["created_at"].isna().sum()
    ),

    "updated_at_nulo_ou_invalido": int(
        df["updated_at"].isna().sum()
    )
}


quality_report = pd.DataFrame(
    list(quality_checks.items()),
    columns=["check", "violacoes"]
)

quality_report["status"] = quality_report["violacoes"].apply(
    lambda x: "PASS" if x == 0 else "FAIL"
)

display(quality_report)

,check,violacoes,status
0,product_id_nulo,0,PASS
1,product_id_duplicado,600,FAIL
2,title_nulo,200,FAIL
3,title_vazio,200,FAIL
4,price_nulo,502,FAIL
5,price_negativo,0,PASS
6,rating_invalido,300,FAIL
7,stock_negativo,250,FAIL
8,created_at_invalido,0,PASS
9,updated_at_nulo_ou_invalido,302,FAIL


In [169]:
import great_expectations as gx

context = gx.get_context()

data_source = context.data_sources.add_pandas("pandas_source")

data_asset = data_source.add_dataframe_asset(
    name="ecommerce_products_raw"
)

batch_definition = data_asset.add_batch_definition_whole_dataframe(
    "whole_dataframe"
)

batch = batch_definition.get_batch(
    batch_parameters={"dataframe": df}
)

print("Batch criado com sucesso!")

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpri27bf7m' for ephemeral docs site


Batch criado com sucesso!


In [170]:
expectations = [
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="product_id"
    ),

    gx.expectations.ExpectColumnValuesToBeUnique(
        column="product_id"
    ),

    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="price"
    ),

    gx.expectations.ExpectColumnValuesToBeBetween(
        column="price",
        min_value=0
    ),

    gx.expectations.ExpectColumnValuesToBeBetween(
        column="rating",
        min_value=1,
        max_value=5
    ),

    gx.expectations.ExpectColumnValuesToBeBetween(
        column="stock_quantity",
        min_value=0
    )
]

for expectation in expectations:
    batch.validate(expectation)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

In [171]:
results = []

for expectation in expectations:
    result = batch.validate(expectation)

    results.append({
        "regra": expectation.__class__.__name__,
        "coluna": expectation.column,
        "passou": result.success
    })

results_df = pd.DataFrame(results)

display(results_df)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

,regra,coluna,passou
0,ExpectColumnValuesToNotBeNull,product_id,True
1,ExpectColumnValuesToBeUnique,product_id,False
2,ExpectColumnValuesToNotBeNull,price,False
3,ExpectColumnValuesToBeBetween,price,True
4,ExpectColumnValuesToBeBetween,rating,False
5,ExpectColumnValuesToBeBetween,stock_quantity,False


In [172]:
results_df["status"] = results_df["passou"].map({
    True: "PASS",
    False: "FAIL"
})

display(
    results_df[["regra", "coluna", "status"]]
)

,regra,coluna,status
0,ExpectColumnValuesToNotBeNull,product_id,PASS
1,ExpectColumnValuesToBeUnique,product_id,FAIL
2,ExpectColumnValuesToNotBeNull,price,FAIL
3,ExpectColumnValuesToBeBetween,price,PASS
4,ExpectColumnValuesToBeBetween,rating,FAIL
5,ExpectColumnValuesToBeBetween,stock_quantity,FAIL


In [173]:
quality_details = pd.DataFrame([
    {
        "Problema": "product_id duplicado",
        "Registros afetados": df["product_id"].duplicated(keep=False).sum()
    },
    {
        "Problema": "price nulo",
        "Registros afetados": df["price"].isna().sum()
    },
    {
        "Problema": "rating inválido",
        "Registros afetados": (~df["rating"].between(1, 5)).sum()
    },
    {
        "Problema": "stock_quantity negativo",
        "Registros afetados": (df["stock_quantity"] < 0).sum()
    },
    {
        "Problema": "title nulo/vazio",
        "Registros afetados": (
            df["title"].isna() |
            df["title"].fillna("").astype(str).str.strip().eq("")
        ).sum()
    },
    {
        "Problema": "updated_at nulo",
        "Registros afetados": df["updated_at"].isna().sum()
    }
])

display(quality_details)

,Problema,Registros afetados
0,product_id duplicado,600
1,price nulo,502
2,rating inválido,300
3,stock_quantity negativo,250
4,title nulo/vazio,200
5,updated_at nulo,302


In [174]:
df_clean = df.copy()


df_clean = df_clean.drop_duplicates(
    subset="product_id",
    keep="first"
).copy()


df_clean["price"] = df_clean["price"].fillna(
    df_clean["price"].median()
)


df_clean.loc[
    ~df_clean["rating"].between(1, 5),
    "rating"
] = pd.NA

df_clean["rating"] = df_clean["rating"].fillna(
    df_clean["rating"].median()
)


df_clean["stock_quantity"] = df_clean[
    "stock_quantity"
].clip(lower=0)

df_clean["title"] = (
    df_clean["title"]
    .fillna("Produto sem título")
    .astype(str)
    .str.strip()
)


df_clean["updated_at"] = df_clean["updated_at"].fillna(
    df_clean["created_at"]
)



print("Registros RAW:", len(df))
print("Registros CLEAN:", len(df_clean))
print("Registros removidos por duplicidade:", len(df) - len(df_clean))

display(df_clean.head())

Registros RAW: 120300
Registros CLEAN: 120000
Registros removidos por duplicidade: 300


,product_id,title,description,category,subcategory,brand,color,locale,seller_type,price,rating,review_count,stock_quantity,views,orders,units_sold,revenue,created_at,updated_at
0,SKU-0000001,Lumi Smartphones Prata Modelo 236,Produto Lumi Smartphones Prata Modelo 236. Fab...,Eletrônicos,Smartphones,Lumi,Prata,en-US,3P,1191.10,4.0,88,40,858,15,15,17866.50,2025-12-27,2026-06-16
1,SKU-0000002,Vitta Colecionáveis Branco Modelo 240,Produto Vitta Colecionáveis Branco Modelo 240....,Games,Colecionáveis,Vitta,Branco,pt-BR,1P,58.63,4.4,91,37,873,51,52,3048.76,2026-05-06,2026-12-17
2,SKU-0000003,Lumi Ficção Verde Modelo 824,Produto Lumi Ficção Verde Modelo 824. Fabricad...,Livros,Ficção,Lumi,Verde,pt-BR,3P,91.96,4.1,93,44,939,28,29,2666.84,2025-12-23,2026-09-30
3,SKU-0000004,PlayCore Ciclismo Rosa Modelo 737,Produto PlayCore Ciclismo Rosa Modelo 737. Fab...,Esportes,Ciclismo,PlayCore,Rosa,en-US,3P,183.71,4.3,96,49,876,17,20,3674.20,2026-04-18,2027-02-09
4,SKU-0000005,UrbanFit Fitness Preto Modelo 498,Produto UrbanFit Fitness Preto Modelo 498. Fab...,Esportes,Fitness,UrbanFit,Preto,en-US,3P,396.34,3.9,79,43,861,56,62,24573.08,2025-02-11,2025-02-18


In [175]:
df_clean.to_csv(
    "ecommerce_products_clean.csv",
    index=False
)

print("Arquivo CLEAN criado com sucesso!")

Arquivo CLEAN criado com sucesso!


In [176]:
quality_details.to_csv(
    "data_quality_raw_report.csv",
    index=False
)

results_df.to_csv(
    "great_expectations_raw_results.csv",
    index=False
)

print("Relatórios de qualidade salvos!")

Relatórios de qualidade salvos!


In [177]:
final_summary = pd.DataFrame([
    {
        "camada": "RAW",
        "registros": len(df),
        "product_id_unicos": df["product_id"].nunique()
    },
    {
        "camada": "CLEAN",
        "registros": len(df_clean),
        "product_id_unicos": df_clean["product_id"].nunique()
    }
])

display(final_summary)

,camada,registros,product_id_unicos
0,RAW,120300,120000
1,CLEAN,120000,120000


In [178]:
from google.colab import files

files.download("ecommerce_products_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [179]:
data_dictionary = pd.DataFrame([
    ["product_id", "Identificador único do produto"],
    ["title", "Nome comercial do produto"],
    ["description", "Descrição textual do produto"],
    ["category", "Categoria principal do produto"],
    ["subcategory", "Subcategoria do produto"],
    ["brand", "Marca do produto"],
    ["color", "Cor do produto"],
    ["locale", "Localidade do produto"],
    ["seller_type", "Tipo de vendedor"],
    ["price", "Preço do produto"],
    ["rating", "Avaliação média do produto"],
    ["review_count", "Quantidade de avaliações"],
    ["stock_quantity", "Quantidade disponível em estoque"],
    ["views", "Quantidade de visualizações"],
    ["orders", "Quantidade de pedidos"],
    ["units_sold", "Quantidade de unidades vendidas"],
    ["revenue", "Receita gerada pelo produto"],
    ["created_at", "Data de criação do registro"],
    ["updated_at", "Data da última atualização"]
], columns=["campo", "descricao"])

display(data_dictionary)

,campo,descricao
0,product_id,Identificador único do produto
1,title,Nome comercial do produto
2,description,Descrição textual do produto
3,category,Categoria principal do produto
4,subcategory,Subcategoria do produto
5,brand,Marca do produto
6,color,Cor do produto
7,locale,Localidade do produto
8,seller_type,Tipo de vendedor
9,price,Preço do produto


In [180]:
data_dictionary.to_csv(
    "data_dictionary.csv",
    index=False
)

In [181]:
from google.colab import files

files.download("data_dictionary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [182]:
import pandas as pd

report = []

for expectation in expectations:
    result = batch.validate(expectation)

    regra = type(expectation).__name__

    coluna = getattr(expectation, "column", None)

    report.append({
        "regra": regra,
        "coluna": coluna,
        "status": "PASS" if result.success else "FAIL"
    })

data_quality_report = pd.DataFrame(report)

display(data_quality_report)

data_quality_report.to_csv(
    "data_quality_report.csv",
    index=False,
    encoding="utf-8"
)

print("Relatório gerado com sucesso!")

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

,regra,coluna,status
0,ExpectColumnValuesToNotBeNull,product_id,PASS
1,ExpectColumnValuesToBeUnique,product_id,FAIL
2,ExpectColumnValuesToNotBeNull,price,FAIL
3,ExpectColumnValuesToBeBetween,price,PASS
4,ExpectColumnValuesToBeBetween,rating,FAIL
5,ExpectColumnValuesToBeBetween,stock_quantity,FAIL


Relatório gerado com sucesso!


In [183]:
from google.colab import files

files.download("data_quality_report.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>